In [64]:
import torch
from torch import nn

In [65]:
batch_size = 2
image_channels = 1
num_classes = 10 # Background 1개 + organ 9개
depth = 32
height = 64
width = 64

# ct_input: [B, C, D, H, W]
ct_input = torch.randn(
    batch_size,
    image_channels,
    depth,
    height,
    width,
)

# logits: [B, K, D, H, W]
logits = torch.randn(
    batch_size,
    num_classes,
    depth,
    height,
    width,
)

# target: [B, D, H, W]
# voxel별 Integer class ID
target = torch.randint(
    low=0,
    high=num_classes,
    size=(batch_size, depth, height, width),
)

# logits    : [B, K, D, H, W]
# prediction: [B, D, H, W]
# 각 voxel에서 가장 높은 logit을 가진 class ID를 선택한다.
prediction = logits.argmax(dim=1)

print("CT input shape: ", ct_input.shape)
print("Logits shape:   ", logits.shape)
print("Target shape:   ", target.shape)
print("Prediction shape:", prediction.shape)
print("Target dtype:   ", target.dtype)
print("Prediction dtype:", prediction.dtype)

# [B, C, D, H, W]
assert ct_input.shape == (
    batch_size,
    image_channels,
    depth,
    height,
    width,
)

# [B, K, D, H, W]
assert logits.shape == (
    batch_size,
    num_classes,
    depth,
    height,
    width,
)

# [B, D, H, W]
assert target.shape == (batch_size, depth, height, width)
assert prediction.shape == target.shape
assert target.dtype == torch.int64
assert prediction.dtype == torch.int64

CT input shape:  torch.Size([2, 1, 32, 64, 64])
Logits shape:    torch.Size([2, 10, 32, 64, 64])
Target shape:    torch.Size([2, 32, 64, 64])
Prediction shape: torch.Size([2, 32, 64, 64])
Target dtype:    torch.int64
Prediction dtype: torch.int64


In [66]:
# 첫 번째 CT patch에서 확인할 voxel 좌표를 선택
batch_index = 0
z_index = 10
y_index = 20
x_index = 30

# [B, K, D, H, W]에서 B, D, H, W를 고정했으므로 tensor.shape = [K]
voxel_logits = logits[
    batch_index,
    :,
    z_index,
    y_index,
    x_index,
]
print("Voxel logits shape:", voxel_logits.shape)

for class_index, score in enumerate(voxel_logits):
    print(f"Class {class_index}: {score.item():.4f}")

# scalar class ID
manual_prediction = voxel_logits.argmax(dim=0)
volume_prediction = prediction[
    batch_index,
    z_index,
    y_index,
    x_index,
]

print("Manual prediction:", manual_prediction.item())
print("Volume prediction:", volume_prediction.item())

# 반드시 같은 class ID를 반환
assert manual_prediction == volume_prediction

# argmax가 실제 maximum logit의 위치를 반환했는지 확인
maximum_logit = voxel_logits.max()
selected_logit = voxel_logits[manual_prediction]

assert selected_logit == maximum_logit


Voxel logits shape: torch.Size([10])
Class 0: 0.1407
Class 1: -0.2810
Class 2: 0.1717
Class 3: -1.5224
Class 4: 1.1944
Class 5: 0.5385
Class 6: 1.7340
Class 7: -2.6698
Class 8: -1.3213
Class 9: -0.2604
Manual prediction: 6
Volume prediction: 6


In [67]:
# Logits: [B, K, D, H, W]

# wrong example
# - argmax는 지정한 dimension을 제거
wrong_prediction = logits.argmax(dim=2)

print("Logits shape:          ", logits.shape)
print("Wrong prediction shape:", wrong_prediction.shape)
print("Target shape:          ", target.shape)

# wrong prediction: [B, K, H, W]
assert wrong_prediction.shape == (
    batch_size,
    num_classes,
    height,
    width,
)

correct_prediction = logits.argmax(dim=1)
print("\nCorrect prediction shape:", correct_prediction.shape)

assert correct_prediction.shape == target.shape


Logits shape:           torch.Size([2, 10, 32, 64, 64])
Wrong prediction shape: torch.Size([2, 10, 64, 64])
Target shape:           torch.Size([2, 32, 64, 64])

Correct prediction shape: torch.Size([2, 32, 64, 64])


In [ ]:
def validate_segmentation_contract(
    ct_volume: torch.Tensor,
    segmentation_logits: torch.Tensor,
    segmentation_target: torch.Tensor,
) -> None:
    """
    Multi-class 3D segmentation의 Tensor Contract validation
    """
    
    # ct_volume: [B, C, D, H, W]
    assert ct_volume.ndim == 5, (
        f"CT input must be 5D, but received {ct_volume.shape}"
    )
    # segmentation_logits: [B, K, D, H, W]
    assert segmentation_logits.ndim == 5, (
        f"Logits must be 5D, but received {segmentation_logits.shape}"
    )
    # segmentation_target: [B, D, H, W]
    assert segmentation_target.ndim == 4, (
        f"Target must be 4D, but received {segmentation_target.shape}"
    )
    
    # ct_input: [B, C, D, H, W]
    batch_size, _, depth, height, width = ct_volume.shape
    
    # logits  : [B, K, D, H, W]
    logits_batch, num_classes, logits_depth, logits_height, logits_width = (
        segmentation_logits.shape
    )

    # B = B / D = D, H = H, W = W   
    assert logits_batch == batch_size
    assert (logits_depth, logits_height, logits_width) == (
        depth,
        height,
        width,
    )

    # Target shape: [B, D, H, W]
    assert segmentation_target.shape == (
        batch_size,
        depth,
        height,
        width,
    )

    # Cross-Entropy용 target은 integer class ID를 저장 (0 ~ 9)
    assert segmentation_target.dtype == torch.int64

    # 모든 target class ID는 [0, K-1] 범위 안에 있어야 한다.
    assert segmentation_target.min().item() >= 0
    assert segmentation_target.max().item() < num_classes



# Test 1) Normal Case
# 정상 Target shape: [B, D, H, W]
validate_segmentation_contract(
    ct_volume=ct_input,
    segmentation_logits=logits,
    segmentation_target=target,
)
print("Valid contract: passed")


# Test 2) Wrong Case
# 비정상 Target shape: [B, 1, D, H, W]
invalid_target = target.unsqueeze(dim=1)

print("test1: Invalid target shape:", invalid_target.shape)

try:
    validate_segmentation_contract(
        ct_volume=ct_input,
        segmentation_logits=logits,
        segmentation_target=invalid_target,
    )
except AssertionError as error:
    print("test2: Invalid contract caught:", error)
else:
    raise RuntimeError("Invalid target was not detected")

Valid contract: passed
test1: Invalid target shape: torch.Size([2, 1, 32, 64, 64])
test2: Invalid contract caught: Target must be 4D, but received torch.Size([2, 1, 32, 64, 64])
